In [3]:
import math
import random
import json
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.signal import find_peaks
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader

# =============================================================================
# 1. Constants & HyperParameters
# =============================================================================
FS_BVP = 64
FS_ACC = 32
FS_SLOW = 4

TASK_TO_CLASS = {'Baseline': 0, 'Logic': 1, 'Sudoku': 1, 'Stroop': 1}
CLASS_NAMES = ['NonStress', 'Stress']

@dataclass
class HyperParameters:
    random_seed: int = 42
    window_seconds: int = 60
    stride_seconds: int = 10
    stride_seconds_eval: int = 60
    batch_size: int = 64
    epochs: int = 50
    learning_rate: float = 0.001
    dropout_rate: float = 0.4
    weight_decay: float = 1e-4
    patience: int = 8
    transformer_d_model: int = 48
    transformer_heads: int = 8
    transformer_layers: int = 2
    transformer_ff_dim: int = 192
    normalization_mode: str = "subject_zscore"
    loss_name: str = "focal"
    focal_gamma: float = 2.0

# =============================================================================
# 2. Data Processing & Loading
# =============================================================================
def find_dataset_root() -> Path:
    return Path('/home/binghin2/Myproject/Dataset/CATSA')

def discover_complete_subjects(root: Path) -> List[str]:
    return sorted([p.name for p in root.glob('Sub*') if p.is_dir()], key=lambda x: int(x[3:]))

def bvp_to_hr_hrv_4hz(bvp_64: np.ndarray, fs: int = 64, out_fs: int = 4):
    sig = np.asarray(bvp_64, dtype=np.float32).reshape(-1)
    peaks, _ = find_peaks(sig, distance=int(fs * 0.30))
    n4 = len(sig) // (fs // out_fs)
    t4 = np.arange(n4) / float(out_fs)
    if len(peaks) < 3:
        return np.zeros(n4, dtype=np.float32), np.zeros(n4, dtype=np.float32)
    t_peaks = peaks / float(fs)
    ibi = np.clip(np.diff(t_peaks), 1e-3, None)
    hr_4 = np.interp(t4, t_peaks[1:], 60.0 / ibi).astype(np.float32)
    hrv = pd.Series(ibi).rolling(window=10, min_periods=1).std().fillna(0).values
    hrv_4 = np.interp(t4, t_peaks[1:], hrv).astype(np.float32)
    return hr_4, hrv_4

def build_split_arrays(dataset_root: Path, subjects: List[str], hp: HyperParameters, is_train: bool = True):
    all_acc, all_slow, all_y = [], [], []
    stride_sec = hp.stride_seconds if is_train else hp.stride_seconds_eval
    len_acc = hp.window_seconds * FS_ACC
    len_slow = hp.window_seconds * FS_SLOW
    stride_slow = stride_sec * FS_SLOW

    for subj in subjects:
        sdir = dataset_root / subj
        for task, label in TASK_TO_CLASS.items():
            tdir = sdir / task
            if not tdir.exists(): continue
            
            try:
                acc = pd.read_csv(tdir / 'ACC.csv').values.astype(np.float32)[:, :3]
                bvp = pd.read_csv(tdir / 'BVP.csv').values.reshape(-1)
                eda = pd.read_csv(tdir / 'EDA.csv').values.reshape(-1)
                temp = pd.read_csv(tdir / 'TEMP.csv').values.reshape(-1)
                
                hr_4, hrv_4 = bvp_to_hr_hrv_4hz(bvp)
                t4 = min(len(eda), len(temp), len(hr_4), len(hrv_4), len(acc)//8)
                if t4 < len_slow: continue
                
                if hp.normalization_mode == "subject_zscore":
                    eda = (eda[:t4] - np.mean(eda[:t4])) / (np.std(eda[:t4]) + 1e-6)
                    temp = (temp[:t4] - np.mean(temp[:t4])) / (np.std(temp[:t4]) + 1e-6)
                    hr_4 = (hr_4[:t4] - np.mean(hr_4[:t4])) / (np.std(hr_4[:t4]) + 1e-6)
                    hrv_4 = (hrv_4[:t4] - np.mean(hrv_4[:t4])) / (np.std(hrv_4[:t4]) + 1e-6)
                    acc_part = acc[:t4*8]
                    acc_part = (acc_part - np.mean(acc_part, axis=0)) / (np.std(acc_part, axis=0) + 1e-6)
                
                slow = np.stack([eda, temp, hr_4, hrv_4], axis=1).astype(np.float32)
                
                start4 = 0
                while start4 + len_slow <= len(slow):
                    all_acc.append(acc_part[start4*8 : start4*8 + len_acc].T.copy())
                    all_slow.append(slow[start4 : start4 + len_slow].T.copy())
                    all_y.append(label)
                    start4 += stride_slow
            except Exception as e:
                continue

    return np.array(all_acc), np.array(all_slow), np.array(all_y, dtype=np.int64)

class MultiModalDataset(Dataset):
    def __init__(self, acc, slow, y):
        self.acc = torch.tensor(acc, dtype=torch.float32)
        self.slow = torch.tensor(slow, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.acc[idx], self.slow[idx], self.y[idx]

def make_loader(acc, slow, y, batch_size: int, shuffle: bool = True) -> DataLoader:
    return DataLoader(MultiModalDataset(acc, slow, y), batch_size=batch_size, shuffle=shuffle, num_workers=2)

# =============================================================================
# 3. Model Architecture
# =============================================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 1024, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class MultiModal1DCNNTransformer(nn.Module):
    def __init__(self, hp: HyperParameters):
        super().__init__()
        self.acc_branch = nn.Sequential(
            nn.Conv1d(3, 16, 9, stride=4, padding=4), nn.BatchNorm1d(16), nn.GELU(), nn.Dropout(hp.dropout_rate),
            nn.Conv1d(16, 32, 9, stride=2, padding=4), nn.BatchNorm1d(32), nn.GELU(), nn.Dropout(hp.dropout_rate)
        )
        self.slow_branch = nn.Sequential(
            nn.Conv1d(4, 16, 5, padding=2), nn.BatchNorm1d(16), nn.GELU(), nn.Dropout(hp.dropout_rate),
            nn.Conv1d(16, 16, 3, padding=1), nn.BatchNorm1d(16), nn.GELU(), nn.Dropout(hp.dropout_rate)
        )
        self.posenc = PositionalEncoding(hp.transformer_d_model, dropout=hp.dropout_rate)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=hp.transformer_d_model, nhead=hp.transformer_heads, 
            dim_feedforward=hp.transformer_ff_dim, dropout=hp.dropout_rate, 
            batch_first=True, norm_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=hp.transformer_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(hp.transformer_d_model)
        self.cls = nn.Linear(hp.transformer_d_model, 2)

    def forward(self, acc, slow):
        fused = torch.cat([self.acc_branch(acc), self.slow_branch(slow)], dim=1)
        x = self.posenc(fused.permute(0, 2, 1).contiguous())
        return self.cls(self.norm(self.transformer(x)).mean(dim=1))

# =============================================================================
# 4. Training & Evaluation
# =============================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        ce = nn.functional.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()

def make_criterion(y_train: np.ndarray, hp: HyperParameters, device: torch.device):
    cnt = np.bincount(y_train, minlength=2)
    cw = torch.tensor(cnt.sum() / (2 * np.maximum(cnt, 1.0)), dtype=torch.float32).to(device)
    criterion = FocalLoss(alpha=cw, gamma=hp.focal_gamma)
    loss_cfg = {"loss_name": hp.loss_name, "focal_gamma": hp.focal_gamma}
    return criterion, loss_cfg

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for acc, slow, y in loader:
        acc, slow, y = acc.to(device), slow.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(acc, slow), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * y.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    preds, trues = [], []
    with torch.no_grad():
        for acc, slow, y in loader:
            acc, slow, y = acc.to(device), slow.to(device), y.to(device)
            logits = model(acc, slow)
            total_loss += criterion(logits, y).item() * y.size(0)
            preds.extend(logits.argmax(1).cpu().numpy())
            trues.extend(y.cpu().numpy())
    
    return {
        "loss": total_loss / max(len(loader.dataset), 1),
        "accuracy": accuracy_score(trues, preds),
        "f1": f1_score(trues, preds, average='macro', zero_division=0),
        "precision": precision_score(trues, preds, average='macro', zero_division=0),
        "recall": recall_score(trues, preds, average='macro', zero_division=0)
    }

# =============================================================================
# 5. Main Execution (Only runs if executed directly, not imported)
# =============================================================================
if __name__ == "__main__":
    hp = HyperParameters()
    random.seed(hp.random_seed)
    np.random.seed(hp.random_seed)
    torch.manual_seed(hp.random_seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(hp.random_seed)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dataset_root = find_dataset_root()
    subjects = discover_complete_subjects(dataset_root)
    
    save_dir = Path("/home/binghin2/Myproject/Research/CATSA/Train/Individual_data/ALL/Save_model")
    save_dir.mkdir(parents=True, exist_ok=True)
    
    all_metrics = []
    
    for fold, test_subj in enumerate(subjects):
        print(f"\n=== Fold {fold+1}/{len(subjects)} | Test: {test_subj} ===")
        val_subj = subjects[(fold + 1) % len(subjects)]
        train_subjs = [s for s in subjects if s not in [test_subj, val_subj]]
        
        x_tr_acc, x_tr_slow, y_tr = build_split_arrays(dataset_root, train_subjs, hp, is_train=True)
        x_va_acc, x_va_slow, y_va = build_split_arrays(dataset_root, [val_subj], hp, is_train=False)
        x_te_acc, x_te_slow, y_te = build_split_arrays(dataset_root, [test_subj], hp, is_train=False)
        
        if len(y_tr) == 0 or len(y_te) == 0: continue
            
        tr_loader = make_loader(x_tr_acc, x_tr_slow, y_tr, hp.batch_size, shuffle=True)
        va_loader = make_loader(x_va_acc, x_va_slow, y_va, hp.batch_size, shuffle=False)
        te_loader = make_loader(x_te_acc, x_te_slow, y_te, hp.batch_size, shuffle=False)
        
        model = MultiModal1DCNNTransformer(hp).to(device)
        criterion, loss_cfg = make_criterion(y_tr, hp, device)
        optimizer = torch.optim.Adam(model.parameters(), lr=hp.learning_rate, weight_decay=hp.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=4)
        
        best_f1, wait, best_epoch = 0.0, 0, -1
        best_state = None
        
        for ep in range(1, hp.epochs + 1):
            train_loss = train_one_epoch(model, tr_loader, optimizer, criterion, device)
            val_metrics = evaluate(model, va_loader, criterion, device)
            scheduler.step(val_metrics["loss"])
            
            if val_metrics["f1"] > best_f1:
                best_f1 = val_metrics["f1"]
                best_epoch = ep
                wait = 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                wait += 1
                if wait >= hp.patience:
                    print(f"Early stopping at epoch {ep} (best epoch: {best_epoch})")
                    break
        
        if best_state is not None:
            model.load_state_dict(best_state)
        
        test_metrics = evaluate(model, te_loader, criterion, device)
        print(f"Fold test metrics: {test_metrics}")
        all_metrics.append(test_metrics)
        
    print("\n=== LOSO Training Complete ===")
    
    if all_metrics:
        keys = all_metrics[0].keys()
        agg_mean = {k: np.mean([m[k] for m in all_metrics]) for k in keys}
        agg_std = {k: np.std([m[k] for m in all_metrics]) for k in keys}
        print(f"Aggregate mean metrics: {agg_mean}")
        print(f"Aggregate std metrics: {agg_std}")
        
        with open(save_dir / "all_1dcnn_transformer_loso_metrics.json", "w") as f:
            json.dump({"mean": agg_mean, "std": agg_std, "hyperparameters": asdict(hp)}, f, indent=2)


=== Fold 1/50 | Test: Sub1 ===
Early stopping at epoch 9 (best epoch: 1)
Fold test metrics: {'loss': 0.2390352189540863, 'accuracy': 0.3333333333333333, 'f1': 0.3142857142857143, 'precision': 0.6363636363636364, 'recall': 0.5555555555555556}

=== Fold 2/50 | Test: Sub2 ===
Early stopping at epoch 12 (best epoch: 4)
Fold test metrics: {'loss': 0.2996528744697571, 'accuracy': 0.5, 'f1': 0.5, 'precision': 0.6666666666666666, 'recall': 0.6666666666666666}

=== Fold 3/50 | Test: Sub3 ===
Early stopping at epoch 9 (best epoch: 1)
Fold test metrics: {'loss': 0.3236970007419586, 'accuracy': 0.16666666666666666, 'f1': 0.14285714285714285, 'precision': 0.09090909090909091, 'recall': 0.3333333333333333}

=== Fold 4/50 | Test: Sub4 ===
Early stopping at epoch 10 (best epoch: 2)
Fold test metrics: {'loss': 0.516061007976532, 'accuracy': 0.25, 'f1': 0.2, 'precision': 0.125, 'recall': 0.5}

=== Fold 5/50 | Test: Sub5 ===
Early stopping at epoch 13 (best epoch: 5)
Fold test metrics: {'loss': 0.346711

In [4]:
# =============================================================================
# 5. Main Execution (폴드별 모델 저장 로직 추가)
# =============================================================================
if __name__ == "__main__":
    hp = HyperParameters()
    random.seed(hp.random_seed)
    np.random.seed(hp.random_seed)
    torch.manual_seed(hp.random_seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(hp.random_seed)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dataset_root = find_dataset_root()
    subjects = discover_complete_subjects(dataset_root)
    
    # 저장 경로 설정
    save_dir = Path("/home/binghin2/Myproject/Research/CATSA/Train/Individual_data/ALL/Save_model")
    save_dir.mkdir(parents=True, exist_ok=True)
    
    all_metrics = []
    
    for fold_idx, test_subj in enumerate(subjects):
        fold_num = fold_idx + 1
        print(f"\n=== Fold {fold_num}/{len(subjects)} | Test: {test_subj} ===")
        
        val_subj = subjects[(fold_idx + 1) % len(subjects)]
        train_subjs = [s for s in subjects if s not in [test_subj, val_subj]]
        
        x_tr_acc, x_tr_slow, y_tr = build_split_arrays(dataset_root, train_subjs, hp, is_train=True)
        x_va_acc, x_va_slow, y_va = build_split_arrays(dataset_root, [val_subj], hp, is_train=False)
        x_te_acc, x_te_slow, y_te = build_split_arrays(dataset_root, [test_subj], hp, is_train=False)
        
        if len(y_tr) == 0 or len(y_te) == 0: continue
            
        tr_loader = make_loader(x_tr_acc, x_tr_slow, y_tr, hp.batch_size, shuffle=True)
        va_loader = make_loader(x_va_acc, x_va_slow, y_va, hp.batch_size, shuffle=False)
        te_loader = make_loader(x_te_acc, x_te_slow, y_te, hp.batch_size, shuffle=False)
        
        model = MultiModal1DCNNTransformer(hp).to(device)
        criterion, loss_cfg = make_criterion(y_tr, hp, device)
        optimizer = torch.optim.Adam(model.parameters(), lr=hp.learning_rate, weight_decay=hp.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=4)
        
        best_f1, wait, best_epoch = 0.0, 0, -1
        best_state = None
        fold_history = []
        
        for ep in range(1, hp.epochs + 1):
            train_loss = train_one_epoch(model, tr_loader, optimizer, criterion, device)
            val_metrics = evaluate(model, va_loader, criterion, device)
            scheduler.step(val_metrics["loss"])
            
            fold_history.append({
                "epoch": ep, "train_loss": train_loss, "val_loss": val_metrics["loss"], "val_f1": val_metrics["f1"]
            })
            
            if val_metrics["f1"] > best_f1:
                best_f1 = val_metrics["f1"]
                best_epoch = ep
                wait = 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                wait += 1
                if wait >= hp.patience:
                    print(f"Early stopping at epoch {ep} (best epoch: {best_epoch})")
                    break
        
        # --- [추가] 폴드별 모델 저장 로직 ---
        if best_state is not None:
            model.load_state_dict(best_state)
            
            # 파일명 규칙: all_1dcnn_transformer_loso_torch_fold_01_Sub1_best.pt
            save_path = save_dir / f"all_1dcnn_transformer_loso_torch_fold_{fold_num:02d}_{test_subj}_best.pt"
            
            payload = {
                "model_state_dict": best_state,
                "fold": fold_num,
                "test_subject": test_subj,
                "best_epoch": best_epoch,
                "hyperparameters": asdict(hp),
                "history": fold_history
            }
            torch.save(payload, save_path)
            print(f"Saved best model for Fold {fold_num} to {save_path}")
        
        test_metrics = evaluate(model, te_loader, criterion, device)
        print(f"Fold {fold_num} test metrics: {test_metrics}")
        all_metrics.append(test_metrics)
        
    print("\n=== ALL LOSO Training Complete ===")
    
    if all_metrics:
        keys = all_metrics[0].keys()
        agg_mean = {k: np.mean([m[k] for m in all_metrics]) for k in keys}
        agg_std = {k: np.std([m[k] for m in all_metrics]) for k in keys}
        
        summary = {
            "mean": agg_mean,
            "std": agg_std,
            "hyperparameters": asdict(hp),
            "all_folds": all_metrics
        }
        
        with open(save_dir / "all_1dcnn_transformer_loso_metrics.json", "w") as f:
            json.dump(summary, f, indent=2)


=== Fold 1/50 | Test: Sub1 ===
Early stopping at epoch 9 (best epoch: 1)
Saved best model for Fold 1 to /home/binghin2/Myproject/Research/CATSA/Train/Individual_data/ALL/Save_model/all_1dcnn_transformer_loso_torch_fold_01_Sub1_best.pt
Fold 1 test metrics: {'loss': 0.23903529345989227, 'accuracy': 0.3333333333333333, 'f1': 0.3142857142857143, 'precision': 0.6363636363636364, 'recall': 0.5555555555555556}

=== Fold 2/50 | Test: Sub2 ===
Early stopping at epoch 12 (best epoch: 4)
Saved best model for Fold 2 to /home/binghin2/Myproject/Research/CATSA/Train/Individual_data/ALL/Save_model/all_1dcnn_transformer_loso_torch_fold_02_Sub2_best.pt
Fold 2 test metrics: {'loss': 0.29968738555908203, 'accuracy': 0.5, 'f1': 0.5, 'precision': 0.6666666666666666, 'recall': 0.6666666666666666}

=== Fold 3/50 | Test: Sub3 ===
Early stopping at epoch 9 (best epoch: 1)
Saved best model for Fold 3 to /home/binghin2/Myproject/Research/CATSA/Train/Individual_data/ALL/Save_model/all_1dcnn_transformer_loso_torc